# FaceSwap on Kaggle GPU (FaceFusion, headless)

Free Kaggle GPU instead of Colab. Before running: **Settings → Accelerator: GPU**, **Internet: ON**. Then run cells top to bottom.

Use only faces you have consent to use. Heads-up: Kaggle restricts deepfake work the same way Colab does (port authors have been banned) — keep usage personal/consensual and light. A cheap RunPod/Vast.ai box is the risk-free option for heavy use.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print(sys.version)

In [ ]:
# 1) Install FaceFusion (official installer, no conda needed here).
#    First run takes a while; later cells reuse it.
%cd /kaggle/working
import os
if not os.path.exists('/kaggle/working/facefusion/facefusion.py'):
    !git clone https://github.com/facefusion/facefusion
else:
    print('facefusion already cloned')
%cd /kaggle/working/facefusion
!python install.py cuda@12 --skip-conda
print('install done')

In [ ]:
# 2) Inputs. Upload files via the Kaggle file panel into /kaggle/working,
#    named exactly source.jpg and target.mp4 -- OR paste Drive share links.
!pip install -q gdown
import os
import gdown

WORK = '/kaggle/working'
SOURCE = f'{WORK}/source.jpg'
TARGET = f'{WORK}/target.mp4'
OUTPUT = f'{WORK}/output.mp4'
SOURCE_URL = ''  # optional: Google Drive share link for the face photo
TARGET_URL = ''  # optional: Google Drive share link for the target video

if SOURCE_URL and not os.path.exists(SOURCE):
    gdown.download(SOURCE_URL, SOURCE, fuzzy=True)
if TARGET_URL and not os.path.exists(TARGET):
    gdown.download(TARGET_URL, TARGET, fuzzy=True)

assert os.path.exists(SOURCE), f'Missing {SOURCE}: upload it or set SOURCE_URL'
assert os.path.exists(TARGET), f'Missing {TARGET}: upload it or set TARGET_URL'
print('inputs OK')

In [ ]:
# 2b) GUI alternative: upload + run + preview in one place.
#     Use this INSTEAD of the Drive-links cell above and the two cells below.
import os
import subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Video

WORK = '/kaggle/working'
FF_DIR = '/kaggle/working/facefusion'
SOURCE = f'{WORK}/source.jpg'
TARGET = f'{WORK}/target.mp4'
OUTPUT = f'{WORK}/output.mp4'

PRESETS = {
    'FAST': ['--processors', 'face_swapper'],
    'BALANCED': ['--processors', 'face_swapper', 'face_enhancer'],
    'QUALITY': ['--processors', 'face_swapper', 'face_enhancer', 'frame_enhancer'],
}

src_up = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False, description='1. Source face')
tgt_up = widgets.FileUpload(accept='.mp4,.mov,.webm', multiple=False, description='2. Target video')
preset_dd = widgets.Dropdown(options=list(PRESETS), value='BALANCED', description='Preset')
run_btn = widgets.Button(description='3. Run face swap', button_style='success')
log = widgets.Output()

def _grab(w):
    v = w.value
    if isinstance(v, dict):  # ipywidgets 7
        _name, meta = next(iter(v.items()))
        return meta['content']
    return v[0]['content']  # ipywidgets 8

def _run(_):
    log.clear_output()
    with log:
        if not src_up.value or not tgt_up.value:
            print('Upload both files first (buttons above).')
            return
        Path(SOURCE).write_bytes(_grab(src_up))
        Path(TARGET).write_bytes(_grab(tgt_up))
        print('files saved, starting FaceFusion (models download on first run)...')
        cmd = (['python', 'facefusion.py', 'headless-run', '-s', SOURCE, '-t', TARGET, '-o', OUTPUT]
               + PRESETS[preset_dd.value]
               + ['--execution-providers', 'cuda', '--output-video-encoder', 'libx264', '--output-video-quality', '80'])
        p = subprocess.Popen(cmd, cwd=FF_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
        p.wait()
        if p.returncode == 0 and os.path.exists(OUTPUT):
            print('done:', OUTPUT)
            display(Video(OUTPUT, embed=True))
            print('Download it from the file panel: /kaggle/working/output.mp4')
        else:
            print('run failed, see log above.')

run_btn.on_click(_run)
display(src_up, tgt_up, preset_dd, run_btn, log)


In [ ]:
# 3) Run the swap. Models auto-download on first run (~1-2 GB, one time per session).
#    FAST = swap only | BALANCED = swap + face enhance | QUALITY = + frame enhance (slow)
PRESET = 'BALANCED'

PRESETS = {
    'FAST': '--processors face_swapper',
    'BALANCED': '--processors face_swapper face_enhancer',
    'QUALITY': '--processors face_swapper face_enhancer frame_enhancer',
}
EXTRA = PRESETS[PRESET]

%cd /kaggle/working/facefusion
!python facefusion.py headless-run -s $SOURCE -t $TARGET -o $OUTPUT $EXTRA --execution-providers cuda --output-video-encoder libx264 --output-video-quality 80
print('done:', OUTPUT)

In [ ]:
# 4) Preview. Download via the file panel: /kaggle/working/output.mp4
from IPython.display import Video
Video('/kaggle/working/output.mp4', embed=True)